# Transformer Example 
## with Korean_Conversation Data from AI-Hub

## Load Data

In [1]:
import os

os.chdir("/kaggle/input/datasets/unyeol/korean-conversation-corpus/한국어대화_new_260226/")
os.getcwd()

'/kaggle/input/datasets/unyeol/korean-conversation-corpus/한국어대화_new_260226'

In [2]:
file_list = sorted(os.listdir())
file_list

['A 음식점(15,726)_new.xlsx',
 'B 의류(15,826)_new.xlsx',
 'C 학원(4,773)_new.xlsx',
 'D 소매점(14,949)_new.xlsx',
 'E 생활서비스(11,087)_new.xlsx',
 'F 카페(7,859)_new.xlsx',
 'G 숙박업(7,113)_new.xlsx',
 'H 관광여가오락(4,949)_new.xlsx',
 'I 부동산(8,131)_new.xlsx',
 'j 교통_최종본(250814).xlsx',
 'k 상수도_최종본(250814).xlsx',
 'l 여권 최종본(250814).xlsx',
 'm 차량등록_최종본(250814).xlsx']

In [5]:
import pandas as pd

all_data_list = []

for file in file_list:
    try:
        df = pd.read_excel(file)
        cols = df.columns
        extracted = pd.DataFrame()

        # [Case A] 'SENTENCE', 'MQ', 'SA' 컬럼 (shift 방식)
        if 'SENTENCE' in cols and 'MQ' in cols and 'SA' in cols:
            df['SA_next'] = df['SA'].shift(-1)
            # 기존 MQ null 체크는 여기서 이미 수행됨
            mask = df['MQ'].notnull() & df['SA_next'].notnull()
            extracted = df.loc[mask, ['MQ', 'SA_next']].copy()
            extracted.columns = ['q', 'a']
            
        # [Case B] 'question'과 '비식별 데이터' 컬럼
        # elif 'question' in cols and '비식별 데이터' in cols:
        #     extracted = df[['question', '비식별 데이터']].copy()
        #     extracted.columns = ['q', 'a']

        if not extracted.empty:
            # 1. NULL 값 제거 (질문이나 답변 중 하나라도 NaN이면 삭제)
            extracted = extracted.dropna(subset=['q', 'a'])

            # 2. 정수(int) 타입 제거 
            # 데이터가 str이 아닌 int인 경우를 찾아 제외합니다.
            is_not_int = extracted['q'].apply(lambda x: not isinstance(x, int)) & \
                         extracted['a'].apply(lambda x: not isinstance(x, int))
            extracted = extracted[is_not_int]

            # 3. '#' 문자 포함 여부 체크 (문자열로 변환 후 체크)
            contains_sharp = extracted['q'].astype(str).str.contains('#', na=False) | \
                             extracted['a'].astype(str).str.contains('#', na=False)
            
            final_extracted = extracted[~contains_sharp]
            all_data_list.append(final_extracted)

    except Exception as e:
        print(f"파일 처리 실패 ({file}): {e}")
        continue

# 3. 모든 리스트를 하나의 DataFrame으로 합치기
if all_data_list:
    final_df = pd.concat(all_data_list, ignore_index=True)
    # 컬럼명 최종 정리
    final_df.columns = ['question', 'answer']
    print(f"총 {len(final_df)}개의 유효한 Q&A 쌍을 추출했습니다.")
else:
    final_df = pd.DataFrame(columns=['question', 'answer'])
    print("추출된 데이터가 없습니다.")

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


총 30797개의 유효한 Q&A 쌍을 추출했습니다.


In [6]:
final_df.head(3)

,question,answer
0,지금 배달되나요?,아 네 배달됩니다
1,짬뽕류는 어떤 게 있나요? 잘 나가는 짬뽕 있나요?,특해물 짬뽕도 있고 전복 새우 짬뽕도 있고 해물 종류도 새우 홍합 전복 없는 게 없습니다
2,전복 들어가는 거는 특해물 짬뽕 시켜야 돼요?,전복 짬뽕 시키면 전복이 들어가죠


In [7]:
final_df.tail(3)

,question,answer
30794,취득세는 몇 퍼센트죠?,1.1퍼센트 내셔야해요
30795,이건 몇 년도에 지었어요?,2008년에 짓고 지금 10년차 되거든요
30796,10년 되었으면 수리도 조금 해야겟다 그죠?,하셔도 되고 안하셔도 되고요


In [8]:
import re

def preprocess_sentence(sentence):
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = sentence.strip()
    return sentence

In [9]:
questions = final_df.question.apply(preprocess_sentence).tolist()
answers = final_df.answer.apply(preprocess_sentence).tolist()

## Create a subword-based Tokenizer

In [10]:
import tensorflow_datasets as tfds

tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    questions + answers, target_vocab_size=2**13
)

2026-03-06 06:48:05.579214: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772779685.600710    1086 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772779685.607255    1086 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772779685.624261    1086 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772779685.624288    1086 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772779685.624291    1086 computation_placer.cc:177] computation placer alr

In [11]:
START_TOKEN = [tokenizer.vocab_size]
END_TOKEN = [tokenizer.vocab_size + 1]
VOCAB_SIZE = tokenizer.vocab_size + 2
MAX_LENGTH = 40

## Define Padding and Tokenization Function

In [12]:
import numpy as np

# 1. 패딩(Padding) 함수: 모든 문장의 길이를 동일하게 맞춤
def pad_seq(seq, maxlen=MAX_LENGTH):
    """
    정수 시퀀스의 길이를 고정된 maxlen으로 맞춥니다.
    - 부족하면 뒤에 0을 채우고, 넘치면 자릅니다.
    """
    if len(seq) < maxlen:
        # 현재 길이보다 부족한 만큼 [0] (보통 패딩 토큰)을 뒤에 이어 붙임
        return seq + [0] * (maxlen - len(seq))
    
    # 설정된 maxlen을 초과하는 경우, 앞에서부터 maxlen만큼만 슬라이싱
    return seq[:maxlen]
    
# 2. 토큰화 및 필터링 함수: 텍스트를 정수 인덱스로 변환하고 패딩 적용
def tokenize_and_filter(inputs, outputs):
    """
    질문(inputs)과 답변(outputs) 쌍을 받아 인코딩 및 패딩을 수행합니다.
    """
    tokenized_inputs, tokenized_outputs = [], []
    
    for (sentence1, sentence2) in zip(inputs, outputs):
        # 각 문장의 시작과 끝에 START_TOKEN과 END_TOKEN을 추가하여 모델에 경계 표시
        # 그 후 tokenizer를 사용해 문장을 정수 시퀀스로 인코딩
        sentence1 = START_TOKEN + tokenizer.encode(sentence1) + END_TOKEN
        sentence2 = START_TOKEN + tokenizer.encode(sentence2) + END_TOKEN
        
        # 처리된 시퀀스를 리스트에 저장
        tokenized_inputs.append(sentence1)
        tokenized_outputs.append(sentence2)
    
    # 모든 시퀀스에 대해 pad_seq 함수를 적용하여 길이를 MAX_LENGTH로 통일
    tokenized_inputs = [pad_seq(seq) for seq in tokenized_inputs]
    tokenized_outputs = [pad_seq(seq) for seq in tokenized_outputs]
    
    # 모델 학습을 위해 최종적으로 NumPy 배열 형태로 반환
    return np.array(tokenized_inputs), np.array(tokenized_outputs)

In [13]:
questions, answers = tokenize_and_filter(questions, answers)

In [14]:
questions

array([[8094,   42, 1380, ...,    0,    0,    0],
       [8094, 1329, 2446, ...,    0,    0,    0],
       [8094, 6502,  482, ...,    0,    0,    0],
       ...,
       [8094, 1760, 4505, ...,    0,    0,    0],
       [8094,  132,   16, ...,    0,    0,    0],
       [8094, 6986, 3218, ...,    0,    0,    0]])

## Load Dataset, DataLoader

In [15]:
from torch.utils.data import Dataset, DataLoader
import torch

class ChatbotDataset(Dataset):
    def __init__(self, question, answer):
        self.question = question
        self.answer  = answer 

    def __len__(self):
        return len(self.question)

    def __getitem__(self, idx):
        return (torch.tensor(self.question[idx], dtype=torch.long), 
                torch.tensor(self.answer[idx][:-1], dtype=torch.long),
                torch.tensor(self.answer[idx][1:], dtype=torch.long))

In [16]:
BATCH_SIZE = 64
data = ChatbotDataset(questions, answers)
dataloader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True)

In [17]:
test = next(iter(dataloader))
test[0].shape

torch.Size([64, 40])

## Define Position Encoding Function

In [18]:
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=9000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

## Define Masking Functions

In [19]:
def create_padding_mask(x):
    # (batch_size, 1, 1, seq_len)
    return (x == 0).unsqueeze(1).unsqueeze(2)


def create_look_ahead_mask(x):
    # (seq_len, seq_len)
    seq_len = x.size(1)
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    if x.is_cuda:
        mask = mask.cuda()
    return mask

## Define Dot_Product Function

In [20]:
def scaled_dot_product_attention(query, key, value, mask=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, -1e9)
    p_attn = F.softmax(scores, dim=-1)
    return torch.matmul(p_attn, value), p_attn

## Define Multi Head Attention Class

In [21]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.depth = d_model // self.num_heads

        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)
    
    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)
        scaled_attention = scaled_attention.transpose(1, 2).contiguous()
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        return self.dense(concat_attention)

## Define Encoder. Decoder Layer

In [22]:
class EncoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x ,  mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2

In [23]:
class DecoderLayer(nn.Module):
    def __init__(self, dff, d_model, num_heads, dropout):
        super(DecoderLayer, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dff),
            nn.ReLU(),
            nn.Linear(dff, d_model)
        )
        self.layernorm1 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(d_model, eps=1e-6)
        self.layernorm3 = nn.LayerNorm(d_model, eps=1e-6)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(x + attn1)

        attn2 = self.mha2(out1, enc_output, enc_output, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(out1 + attn2)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        out3 = self.layernorm3(out2 + ffn_output)
        return out3

## Define Trnasformer Class

In [24]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, num_layers, dff, d_model, num_heads, dropout):
        super(Transformer, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        
        self.enc_layers = nn.ModuleList([EncoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderLayer(dff, d_model, num_heads, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)
    
    def forward(self, inputs, dec_inputs):
        # 1. 마스크 생성
        enc_padding_mask = create_padding_mask(inputs)
        dec_padding_mask = create_padding_mask(inputs)
        look_ahead_mask = torch.max(
            create_look_ahead_mask(dec_inputs),
            create_padding_mask(dec_inputs)
        )

        # 2. 인코더
        enc_out = self.embedding(inputs) * math.sqrt(self.d_model)
        enc_out = self.dropout(self.pos_encoding(enc_out))
        for layer in self.enc_layers:
            enc_out = layer(enc_out, enc_padding_mask)
        

        # 3. 디코더
        dec_out = self.embedding(dec_inputs) * math.sqrt(self.d_model)
        dec_out = self.dropout(self.pos_encoding(dec_out))
        for layer in self.dec_layers:
            dec_out = layer(dec_out, enc_out, look_ahead_mask, dec_padding_mask)

        return self.fc_out(dec_out)


## Hyperparameter Setting

In [25]:
NUM_LAYERS = 2
D_MODEL = 256
NUM_HEADS = 8
DFF = 512
DROPOUT = 0.1
EPOCHS = 50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [26]:
class CustomSchedule:
    def __init__(self, d_model, warmup_steps=4000):
        self.d_model = d_model
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = step + 1 # 0으로 나누는 것을 방지
        arg1 = step ** -0.5
        arg2 = step * (self.warmup_steps ** -1.5)
        return (self.d_model ** -0.5) * min(arg1, arg2)

In [27]:
import math
model = Transformer(VOCAB_SIZE, NUM_LAYERS, DFF, D_MODEL, NUM_HEADS, DROPOUT).to(device) # Model -> Transformer

criterion = nn.CrossEntropyLoss(ignore_index=0) # Criterion
optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9) # Optimizer
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=CustomSchedule(D_MODEL)) # Scheduler

## Training

In [28]:
import torch.nn.functional as F
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, dec_inputs, outputs) in enumerate(dataloader):
        inputs, dec_inputs, outputs = inputs.to(device), dec_inputs.to(device), outputs.to(device)
        
        optimizer.zero_grad()
        predictions = model(inputs, dec_inputs)
        
        # predictions shape: (batch_size, seq_len, vocab_size)
        # outputs shape: (batch_size, seq_len)
        loss = criterion(predictions.view(-1, VOCAB_SIZE), outputs.view(-1))
        
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(dataloader):.4f}")


Epoch 1/50 Loss: 7.5285
Epoch 2/50 Loss: 6.3596
Epoch 3/50 Loss: 5.8402
Epoch 4/50 Loss: 5.3765
Epoch 5/50 Loss: 4.9780
Epoch 6/50 Loss: 4.6178
Epoch 7/50 Loss: 4.3047
Epoch 8/50 Loss: 4.0282
Epoch 9/50 Loss: 3.7813
Epoch 10/50 Loss: 3.4745
Epoch 11/50 Loss: 3.1837
Epoch 12/50 Loss: 2.9158
Epoch 13/50 Loss: 2.6807
Epoch 14/50 Loss: 2.4672
Epoch 15/50 Loss: 2.2815
Epoch 16/50 Loss: 2.1161
Epoch 17/50 Loss: 1.9656
Epoch 18/50 Loss: 1.8445
Epoch 19/50 Loss: 1.7228
Epoch 20/50 Loss: 1.6246
Epoch 21/50 Loss: 1.5309
Epoch 22/50 Loss: 1.4510
Epoch 23/50 Loss: 1.3802
Epoch 24/50 Loss: 1.3087
Epoch 25/50 Loss: 1.2537
Epoch 26/50 Loss: 1.2023
Epoch 27/50 Loss: 1.1513
Epoch 28/50 Loss: 1.1116
Epoch 29/50 Loss: 1.0659
Epoch 30/50 Loss: 1.0311
Epoch 31/50 Loss: 0.9987
Epoch 32/50 Loss: 0.9711
Epoch 33/50 Loss: 0.9368
Epoch 34/50 Loss: 0.9091
Epoch 35/50 Loss: 0.8825
Epoch 36/50 Loss: 0.8605
Epoch 37/50 Loss: 0.8414
Epoch 38/50 Loss: 0.8159
Epoch 39/50 Loss: 0.7929
Epoch 40/50 Loss: 0.7768
Epoch 41/

## Evaluation + Prediction

In [29]:
def evaluate(sentence):
    sentence = preprocess_sentence(sentence)
    
    # 수정 포인트: START_TOKEN과 END_TOKEN을 감싸고 있던 대괄호 [] 제거
    sentence_tensor = torch.tensor(START_TOKEN + tokenizer.encode(sentence) + END_TOKEN).unsqueeze(0).to(device)
    output_tensor = torch.tensor(START_TOKEN).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        for i in range(MAX_LENGTH):
            predictions = model(sentence_tensor, output_tensor)
            
            # 현재(마지막) 시점의 예측 단어를 가져옵니다.
            predictions = predictions[:, -1:, :]
            predicted_id = torch.argmax(predictions, dim=-1)

            if predicted_id.item() == END_TOKEN[0]:
                break

            output_tensor = torch.cat([output_tensor, predicted_id], dim=-1)

    return output_tensor.squeeze(0).cpu().numpy()

In [30]:
def predict(sentence):
    prediction = evaluate(sentence)
    predicted_sentence = tokenizer.decode(
        [i for i in prediction if i < tokenizer.vocab_size])
    
    print('Input: {}'.format(sentence))
    print('Output: {}'.format(predicted_sentence))
    return predicted_sentence

In [37]:
print(predict("중학교 3학년 영어 학원비가 얼마죠?"))
print(predict("짜장면 한 그릇이 얼마죠?"))
print(predict("용산구 평균 아파트 가격이 얼마죠?"))

Input: 중학교 3학년 영어 학원비가 얼마죠?
Output: 이십삼만 원입니다
이십삼만 원입니다
Input: 짜장면 한 그릇이 얼마죠?
Output: 4500원입니다
4500원입니다
Input: 용산구 평균 아파트 가격이 얼마죠?
Output: 3억입니다
3억입니다


In [40]:
os.getcwd()

'/kaggle/input/datasets/unyeol/korean-conversation-corpus/한국어대화_new_260226'

In [42]:
torch.save(model.state_dict(), "/kaggle/working/ko-con.pth") # 학습 결과 저장
tokenizer.save_to_file("/kaggle/working/ko-con_tokenizer")